# Introduccion
Notebook de auditoria/reproducibilidad. Se limpiaron outputs para portabilidad sin modificar calculos.


In [ ]:
# Notebook 06 — MLOps and Experiment Management Audit (Level 6)



**Purpose:** This notebook consolidates experiment traceability, model versioning, reproducibility checks, assumptions, and limitations for the Minsur project.



This notebook is an **audit notebook**:

- It does **not retrain models**.

- It does **not recompute SHAP or what-if scenarios**.

- It audits existing artifacts produced by notebooks 01–04.



Tracking policy:

- **Primary tracking:** MLflow runs and metrics.

- **Reproducible fallback:** local CSV/JSON artifacts in `reports/metrics/` and `models/selected/`.



Audit objective:

1. Track experiments, parameters, and metrics.

2. Manage model versions and identify selected model.

3. Validate reproducibility for training/inference artifacts.

4. Document assumptions, configuration constraints, and limitations.


In [ ]:
import os

import json

from pathlib import Path

from datetime import datetime, timezone



import pandas as pd

import mlflow

from mlflow.tracking import MlflowClient



# Robust project root resolution (works from notebooks/ or project root)

CANDIDATES = [Path.cwd(), Path.cwd().parent]

PROJECT_ROOT = next((p for p in CANDIDATES if (p / 'config.yaml').exists()), Path.cwd())

os.chdir(PROJECT_ROOT)



REPORTS_METRICS_DIR = PROJECT_ROOT / 'reports' / 'metrics'

REPORTS_FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'

MODELS_SELECTED_DIR = PROJECT_ROOT / 'models' / 'selected'

MLRUNS_DIR = PROJECT_ROOT / 'mlruns'



REPORTS_METRICS_DIR.mkdir(parents=True, exist_ok=True)



audit_timestamp_utc = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')



# MLflow 3.x file-store compatibility for local prototype tracking

os.environ.setdefault('MLFLOW_ALLOW_FILE_STORE', 'true')



try:

    mlflow.set_tracking_uri(str(MLRUNS_DIR))

except Exception:

    pass



tracking_uri = mlflow.get_tracking_uri()

mlflow_version = mlflow.__version__



all_run_rows = []

experiments_count = 0

total_runs_count = 0

mlflow_access_ok = True

mlflow_warning = ''



try:

    client = MlflowClient()

    experiments = client.search_experiments()

    experiments_count = len(experiments)



    for exp in experiments:

        runs = client.search_runs(

            experiment_ids=[exp.experiment_id],

            max_results=5000,

            order_by=['start_time DESC'],

        )

        for run in runs:

            metrics = run.data.metrics

            params = run.data.params

            tags = run.data.tags

            run_name = tags.get('mlflow.runName', '')

            scenario = params.get('scenario', tags.get('scenario', ''))

            model_name = params.get('model_name', tags.get('model_name', ''))



            all_run_rows.append({

                'experiment_id': exp.experiment_id,

                'experiment_name': exp.name,

                'run_id': run.info.run_id,

                'run_name': run_name,

                'status': run.info.status,

                'model_name': model_name,

                'scenario': scenario,

                'val_RMSE': metrics.get('val_RMSE', None),

                'val_R2': metrics.get('val_R2', None),

                'test_RMSE': metrics.get('test_RMSE', None),

                'test_R2': metrics.get('test_R2', None),

                'artifact_uri': run.info.artifact_uri,

            })

except Exception as exc:

    mlflow_access_ok = False

    mlflow_warning = f'MLflow query failed. Falling back to local artifact audit only. Reason: {exc}'



runs_df = pd.DataFrame(all_run_rows)

total_runs_count = len(runs_df)



audit_overview = pd.DataFrame([

    {

        'audit_timestamp_utc': audit_timestamp_utc,

        'mlflow_version': mlflow_version,

        'tracking_uri': tracking_uri,

        'experiments_count': experiments_count,

        'total_runs_count': total_runs_count,

        'mlflow_access_ok': mlflow_access_ok,

    }

])



print('Audit overview:')

display(audit_overview)



if (not mlflow_access_ok) or runs_df.empty:

    if mlflow_warning:

        print(f'WARNING: {mlflow_warning}')

    else:

        print('WARNING: MLflow has no runs in the configured tracking store. Falling back to local artifact audit only.')

    runs_audit = pd.DataFrame(columns=[

        'experiment_name', 'run_name', 'status', 'model_name', 'scenario',

        'val_RMSE', 'val_R2', 'test_RMSE', 'test_R2', 'artifact_uri'

    ])

else:

    runs_audit = runs_df[[

        'experiment_name', 'run_name', 'status', 'model_name', 'scenario',

        'val_RMSE', 'val_R2', 'test_RMSE', 'test_R2', 'artifact_uri'

    ]].copy()

    runs_audit = runs_audit.sort_values(['experiment_name', 'run_name'], na_position='last').reset_index(drop=True)



runs_audit_path = REPORTS_METRICS_DIR / 'mlflow_runs_audit.csv'

runs_audit.to_csv(runs_audit_path, index=False)



print(f'MLflow runs audit saved to: {runs_audit_path}')

display(runs_audit.head(30))


In [ ]:
# MLOps leaderboard (training/modeling vs explainability vs what-if)

def classify_run_type(run_name: str, scenario: str, experiment_name: str) -> str:

    text = ' '.join([str(run_name), str(scenario), str(experiment_name)]).lower()

    if 'what_if' in text or 'simulation' in text:

        return 'what_if'

    if 'explain' in text or 'shap' in text:

        return 'explainability'

    return 'training_modeling'



leaderboard_df = runs_audit.copy()

if not leaderboard_df.empty:

    leaderboard_df['run_type'] = leaderboard_df.apply(

        lambda row: classify_run_type(row.get('run_name', ''), row.get('scenario', ''), row.get('experiment_name', '')),

        axis=1,

    )

    leaderboard_df['val_RMSE_sort'] = pd.to_numeric(leaderboard_df['val_RMSE'], errors='coerce')

    leaderboard_df['test_RMSE_sort'] = pd.to_numeric(leaderboard_df['test_RMSE'], errors='coerce')

    leaderboard_df = leaderboard_df.sort_values(

        by=['val_RMSE_sort', 'test_RMSE_sort'],

        ascending=[True, True],

        na_position='last',

    ).reset_index(drop=True)

    leaderboard_df['rank'] = leaderboard_df.index + 1

    leaderboard_df = leaderboard_df[[

        'rank', 'run_type', 'experiment_name', 'run_name', 'status', 'model_name', 'scenario',

        'val_RMSE', 'val_R2', 'test_RMSE', 'test_R2', 'artifact_uri'

    ]]

else:

    leaderboard_df = pd.DataFrame(columns=[

        'rank', 'run_type', 'experiment_name', 'run_name', 'status', 'model_name', 'scenario',

        'val_RMSE', 'val_R2', 'test_RMSE', 'test_R2', 'artifact_uri'

    ])



leaderboard_path = REPORTS_METRICS_DIR / 'mlflow_leaderboard_audit.csv'

leaderboard_df.to_csv(leaderboard_path, index=False)



print(f'MLflow leaderboard audit saved to: {leaderboard_path}')

display(leaderboard_df.head(40))


In [ ]:
# Selected model metadata summary

selected_metadata_path = MODELS_SELECTED_DIR / 'selected_model_metadata.json'

if not selected_metadata_path.exists():

    raise FileNotFoundError('Missing models/selected/selected_model_metadata.json. Run notebook 02 first.')



with open(selected_metadata_path, 'r', encoding='utf-8') as f:

    selected_metadata = json.load(f)



def extract_model_row(role_key: str, role_label: str) -> dict:

    payload = selected_metadata.get(role_key, {})

    val_metrics = payload.get('validation_metrics', {})

    test_metrics = payload.get('test_metrics', {})

    return {

        'role': role_label,

        'scenario': payload.get('scenario', ''),

        'model_name': payload.get('model_name', ''),

        'validation_MAE': val_metrics.get('MAE', None),

        'validation_RMSE': val_metrics.get('RMSE', None),

        'validation_R2': val_metrics.get('R2', None),

        'test_MAE': test_metrics.get('MAE', None),

        'test_RMSE': test_metrics.get('RMSE', None),

        'test_R2': test_metrics.get('R2', None),

        'uses_feed_variables': payload.get('uses_feed_variables', None),

        'uses_target_lags': payload.get('uses_target_lags', None),

        'model_path': payload.get('model_path', selected_metadata.get('saved_model_path', '')),

        'feature_columns_path': payload.get('feature_columns_path', selected_metadata.get('feature_columns_path', '')),

        'note': payload.get('note', ''),

    }



metadata_summary_df = pd.DataFrame([

    extract_model_row('best_statistical_model', 'best_statistical_model'),

    extract_model_row('recommended_model_with_lagged_lab_assumption', 'recommended_model_with_lagged_lab_assumption'),

    extract_model_row('strict_no_lab_input_fallback', 'strict_no_lab_input_fallback'),

])



metadata_summary_path = REPORTS_METRICS_DIR / 'selected_model_metadata_audit.csv'

metadata_summary_df.to_csv(metadata_summary_path, index=False)



print(f'Selected metadata audit saved to: {metadata_summary_path}')

display(metadata_summary_df)


In [ ]:
# Formal model versioning table

def model_status_from_name(model_file: str) -> str:

    name = model_file.lower()

    if name == 'model_lagged_lab_assumption.pkl':

        return 'selected'

    if name == 'model_strict_no_lab_input.pkl':

        return 'fallback'

    if any(k in name for k in ['lag_1_available', 'lag_3_available', 'lag_6_available', 'no_recent_lab_available']):

        return 'sensitivity'

    if name == 'model.pkl':

        return 'legacy'

    return 'unknown'



def role_from_name(model_file: str) -> str:

    name = model_file.lower()

    if name == 'model_lagged_lab_assumption.pkl':

        return 'recommended_model_with_lagged_lab_assumption'

    if name == 'model_strict_no_lab_input.pkl':

        return 'strict_no_lab_input_fallback'

    if name == 'model.pkl':

        return 'legacy_selected_alias'

    if 'lag_' in name and 'available' in name:

        return 'lab_delay_sensitivity_model'

    return 'unclassified'



def scenario_from_name(model_file: str) -> str:

    name = model_file.lower()

    if 'lag_1_available' in name:

        return 'lag_1_available'

    if 'lag_3_available' in name:

        return 'lag_3_available'

    if 'lag_6_available' in name:

        return 'lag_6_available'

    if 'no_recent_lab_available' in name:

        return 'no_recent_lab_available'

    if name == 'model_lagged_lab_assumption.pkl':

        return selected_metadata.get('recommended_model_with_lagged_lab_assumption', {}).get('scenario', '')

    if name == 'model_strict_no_lab_input.pkl':

        return selected_metadata.get('strict_no_lab_input_fallback', {}).get('scenario', '')

    return ''



def allowed_lags_from_scenario(scenario: str) -> str:

    mapping = {

        'lag_1_available': '1,3,6,12',

        'lag_3_available': '3,6,12',

        'lag_6_available': '6,12',

        'no_recent_lab_available': 'none',

    }

    return mapping.get(scenario, '')



model_rows = []

for model_file in sorted(MODELS_SELECTED_DIR.glob('*.pkl')):

    model_name = model_file.name

    scenario_name = scenario_from_name(model_name)

    meta_file = None

    meta_payload = {}



    if scenario_name in {'lag_1_available', 'lag_3_available', 'lag_6_available', 'no_recent_lab_available'}:

        candidate = MODELS_SELECTED_DIR / f'{scenario_name}_best_model_metadata.json'

        if candidate.exists():

            meta_file = candidate

    elif model_name == 'model_lagged_lab_assumption.pkl':

        meta_file = selected_metadata_path

    elif model_name == 'model_strict_no_lab_input.pkl':

        meta_file = selected_metadata_path



    if meta_file is not None and meta_file.exists():

        with open(meta_file, 'r', encoding='utf-8') as f:

            meta_payload = json.load(f)



    val_rmse = None

    val_r2 = None

    test_rmse = None

    test_r2 = None

    feature_file = ''

    op_assumption = ''



    if model_name == 'model_lagged_lab_assumption.pkl':

        role_payload = selected_metadata.get('recommended_model_with_lagged_lab_assumption', {})

        val_rmse = role_payload.get('validation_metrics', {}).get('RMSE', None)

        val_r2 = role_payload.get('validation_metrics', {}).get('R2', None)

        test_rmse = role_payload.get('test_metrics', {}).get('RMSE', None)

        test_r2 = role_payload.get('test_metrics', {}).get('R2', None)

        feature_file = role_payload.get('feature_columns_path', str(MODELS_SELECTED_DIR / 'feature_columns.json'))

        op_assumption = 'Latest lagged lab result available at inference time.'

    elif model_name == 'model_strict_no_lab_input.pkl':

        role_payload = selected_metadata.get('strict_no_lab_input_fallback', {})

        val_rmse = role_payload.get('validation_metrics', {}).get('RMSE', None)

        val_r2 = role_payload.get('validation_metrics', {}).get('R2', None)

        test_rmse = role_payload.get('test_metrics', {}).get('RMSE', None)

        test_r2 = role_payload.get('test_metrics', {}).get('R2', None)

        feature_file = role_payload.get('feature_columns_path', str(MODELS_SELECTED_DIR / 'feature_columns_strict_no_lab_input.json'))

        op_assumption = 'No lagged laboratory quality inputs allowed.'

    elif scenario_name in {'lag_1_available', 'lag_3_available', 'lag_6_available', 'no_recent_lab_available'}:

        val_rmse = meta_payload.get('validation_metrics', {}).get('RMSE', None)

        val_r2 = meta_payload.get('validation_metrics', {}).get('R2', None)

        if not runs_audit.empty:

            match_runs = runs_audit[runs_audit['scenario'] == scenario_name]

            if not match_runs.empty:

                test_rmse = pd.to_numeric(match_runs['test_RMSE'], errors='coerce').dropna().iloc[0] if pd.to_numeric(match_runs['test_RMSE'], errors='coerce').notna().any() else None

                test_r2 = pd.to_numeric(match_runs['test_R2'], errors='coerce').dropna().iloc[0] if pd.to_numeric(match_runs['test_R2'], errors='coerce').notna().any() else None

        feature_file = 'feature_columns in scenario metadata'

        op_assumption = meta_payload.get('availability_assumption', 'Scenario-specific lab-delay assumption.')



    model_rows.append({

        'model_file': model_name,

        'role': role_from_name(model_name),

        'scenario': scenario_name,

        'allowed_lags': allowed_lags_from_scenario(scenario_name),

        'validation_RMSE': val_rmse,

        'validation_R2': val_r2,

        'test_RMSE': test_rmse,

        'test_R2': test_r2,

        'feature_file': feature_file,

        'metadata_file': str(meta_file) if meta_file else '',

        'operational_assumption': op_assumption,

        'status': model_status_from_name(model_name),

    })



model_version_audit_df = pd.DataFrame(model_rows)

model_version_audit_path = REPORTS_METRICS_DIR / 'model_version_audit.csv'

model_version_audit_df.to_csv(model_version_audit_path, index=False)



print(f'Model version audit saved to: {model_version_audit_path}')

display(model_version_audit_df)


In [ ]:
# Reproducibility artifact checklist + model-feature consistency

artifacts_to_check = [

    {

        'artifact': 'recommended_model',

        'expected_path': str(MODELS_SELECTED_DIR / 'model_lagged_lab_assumption.pkl'),

        'purpose': 'Primary model for inference under lagged-lab assumption.',

        'risk_if_missing': 'No deployable primary model.',

    },

    {

        'artifact': 'recommended_feature_list',

        'expected_path': str(MODELS_SELECTED_DIR / 'feature_columns.json'),

        'purpose': 'Exact inference schema for primary model.',

        'risk_if_missing': 'Inference schema mismatch risk.',

    },

    {

        'artifact': 'selected_model_metadata',

        'expected_path': str(MODELS_SELECTED_DIR / 'selected_model_metadata.json'),

        'purpose': 'Selection rationale, metrics, assumptions.',

        'risk_if_missing': 'Model selection is not auditable.',

    },

    {

        'artifact': 'strict_fallback_model',

        'expected_path': str(MODELS_SELECTED_DIR / 'model_strict_no_lab_input.pkl'),

        'purpose': 'Conservative fallback without lab lags.',

        'risk_if_missing': 'No strict operational fallback.',

    },

    {

        'artifact': 'strict_fallback_feature_list',

        'expected_path': str(MODELS_SELECTED_DIR / 'feature_columns_strict_no_lab_input.json'),

        'purpose': 'Exact inference schema for fallback model.',

        'risk_if_missing': 'Fallback inference schema mismatch risk.',

    },

    {

        'artifact': 'baseline_metrics',

        'expected_path': str(REPORTS_METRICS_DIR / 'baseline_metrics.json'),

        'purpose': 'Baseline performance reference.',

        'risk_if_missing': 'No benchmark for model lift.',

    },

    {

        'artifact': 'scenario_parameters',

        'expected_path': str(REPORTS_METRICS_DIR / 'scenario_parameters.json'),

        'purpose': 'What-if scenario parameter traceability.',

        'risk_if_missing': 'Scenario analysis is not reproducible.',

    },

    {

        'artifact': 'top_shap_features',

        'expected_path': str(REPORTS_METRICS_DIR / 'top_shap_features.csv'),

        'purpose': 'Explainability driver evidence.',

        'risk_if_missing': 'No traceable global explainability outputs.',

    },

    {

        'artifact': 'scenario_results',

        'expected_path': str(REPORTS_METRICS_DIR / 'scenario_results.csv'),

        'purpose': 'What-if scenario outcome table.',

        'risk_if_missing': 'No scenario impact audit trail.',

    },

    {

        'artifact': 'experiment_log',

        'expected_path': str(REPORTS_METRICS_DIR / 'experiment_log.csv'),

        'purpose': 'Local experiment fallback log.',

        'risk_if_missing': 'Reduced reproducibility when MLflow is unavailable.',

    },

]



repro_rows = []

for item in artifacts_to_check:

    p = Path(item['expected_path'])

    repro_rows.append({

        'artifact': item['artifact'],

        'expected_path': str(p),

        'exists': p.exists(),

        'purpose': item['purpose'],

        'risk_if_missing': item['risk_if_missing'],

    })



reproducibility_checklist_df = pd.DataFrame(repro_rows)

reproducibility_checklist_path = REPORTS_METRICS_DIR / 'reproducibility_artifact_checklist.csv'

reproducibility_checklist_df.to_csv(reproducibility_checklist_path, index=False)



with open(MODELS_SELECTED_DIR / 'feature_columns.json', 'r', encoding='utf-8') as f:

    main_features = json.load(f)

with open(MODELS_SELECTED_DIR / 'feature_columns_strict_no_lab_input.json', 'r', encoding='utf-8') as f:

    fallback_features = json.load(f)



main_has_target_lags = any('silica concentrate_lag_' in col.lower() for col in main_features)

fallback_has_target_lags = any('silica concentrate_lag_' in col.lower() for col in fallback_features)



feature_consistency_df = pd.DataFrame([

    {

        'check': 'main_feature_count',

        'value': len(main_features),

        'expected': '> 0',

        'status': 'pass' if len(main_features) > 0 else 'fail',

    },

    {

        'check': 'fallback_feature_count',

        'value': len(fallback_features),

        'expected': '> 0',

        'status': 'pass' if len(fallback_features) > 0 else 'fail',

    },

    {

        'check': 'main_includes_target_lags',

        'value': main_has_target_lags,

        'expected': True,

        'status': 'pass' if main_has_target_lags else 'fail',

    },

    {

        'check': 'fallback_excludes_target_lags',

        'value': (not fallback_has_target_lags),

        'expected': True,

        'status': 'pass' if not fallback_has_target_lags else 'fail',

    },

])



feature_consistency_path = REPORTS_METRICS_DIR / 'feature_consistency_audit.csv'

feature_consistency_df.to_csv(feature_consistency_path, index=False)



print(f'Reproducibility checklist saved to: {reproducibility_checklist_path}')

display(reproducibility_checklist_df)

print(f'Feature consistency audit saved to: {feature_consistency_path}')

display(feature_consistency_df)


In [ ]:
# Assumptions, limitations, roadmap, Level 6 checklist, and markdown summary

assumptions_df = pd.DataFrame([

    {

        'assumption': 'Lagged lab availability assumption',

        'where_documented': 'selected_model_metadata.json, notebooks 02/03/04',

        'why_it_matters': 'Primary model validity depends on latest lab result being available.',

        'risk': 'If lab delay is longer, inference assumptions break.',

        'mitigation': 'Use strict fallback and validate real lab turnaround with operations.',

    },

    {

        'assumption': 'Hourly modeling frequency',

        'where_documented': 'config.yaml:data.modeling_frequency',

        'why_it_matters': 'Defines temporal granularity for features and lags.',

        'risk': 'Mismatched operational cadence reduces reliability.',

        'mitigation': 'Align inference schedule to 1h windows.',

    },

    {

        'assumption': 'Feed variable availability',

        'where_documented': 'selected_model_metadata.json:uses_feed_variables',

        'why_it_matters': 'Main model uses feed chemistry features.',

        'risk': 'Missing feed variables causes schema/inference failures.',

        'mitigation': 'Monitor feed data pipelines and fallback when unavailable.',

    },

    {

        'assumption': 'No contemporaneous target leakage',

        'where_documented': 'feature_engineering.py (lag-only target usage)',

        'why_it_matters': 'Prevents leakage and inflated metrics.',

        'risk': 'Using current target at inference is invalid.',

        'mitigation': 'Keep only lagged target inputs and schema validation.',

    },

    {

        'assumption': 'Sensor-only fallback weakness',

        'where_documented': 'selected_model_metadata.json strict fallback metrics',

        'why_it_matters': 'Fallback has lower predictive power.',

        'risk': 'Lower quality predictions under strict constraints.',

        'mitigation': 'Treat as conservative fallback and improve sensor feature set.',

    },

    {

        'assumption': 'What-if simulations are predictive, not causal',

        'where_documented': 'notebook 04 disclaimer and conclusions',

        'why_it_matters': 'Avoids misuse as control policy.',

        'risk': 'Operational decisions based on causal misinterpretation.',

        'mitigation': 'Use as decision-support with expert validation.',

    },

    {

        'assumption': 'MLflow local tracking limitation',

        'where_documented': 'config.yaml mlruns local URI',

        'why_it_matters': 'No centralized multi-user governance by default.',

        'risk': 'Limited collaboration and retention controls.',

        'mitigation': 'Migrate MLflow backend to managed DB/server.',

    },

    {

        'assumption': 'Model Registry not implemented in prototype',

        'where_documented': 'project scope and current artifacts',

        'why_it_matters': 'No formal stage transitions/aliases.',

        'risk': 'Manual model promotion and version ambiguity.',

        'mitigation': 'Formalize Model Registry with stages and aliases.',

    },

])

assumptions_path = REPORTS_METRICS_DIR / 'operational_assumptions_audit.csv'

assumptions_df.to_csv(assumptions_path, index=False)



limitations_df = pd.DataFrame([

    {'limitation': 'MLflow is local, not centralized.'},

    {'limitation': 'No formal Model Registry with stages/aliases.'},

    {'limitation': 'No automated retraining pipeline.'},

    {'limitation': 'No online drift monitoring in production.'},

    {'limitation': 'Operational validity depends on real lab delay behavior.'},

    {'limitation': 'Sensor-only fallback performance is weak relative to primary model.'},

    {'limitation': 'What-if scenarios are predictive and not causal recommendations.'},

])

limitations_path = REPORTS_METRICS_DIR / 'mlops_limitations_audit.csv'

limitations_df.to_csv(limitations_path, index=False)



roadmap_df = pd.DataFrame([

    {

        'next_step': 'Migrate MLflow to SQLite/PostgreSQL backend',

        'priority': 'High',

        'reason': 'Enable robust persistence and concurrent tracking.',

        'owner_candidate': 'ML Engineer',

        'expected_output': 'Centralized, stable MLflow tracking backend.',

    },

    {

        'next_step': 'Formalize Model Registry',

        'priority': 'High',

        'reason': 'Add controlled promotion/version governance.',

        'owner_candidate': 'MLOps Engineer',

        'expected_output': 'Registered model with stage/alias workflow.',

    },

    {

        'next_step': 'Validate lab delay with operations',

        'priority': 'High',

        'reason': 'Primary model relies on lagged lab availability.',

        'owner_candidate': 'Process + Data Team',

        'expected_output': 'Documented SLA for lab turnaround.',

    },

    {

        'next_step': 'Build inference pipeline',

        'priority': 'High',

        'reason': 'Operationalize reproducible scoring.',

        'owner_candidate': 'Data Engineer',

        'expected_output': 'Scheduled/streaming inference job with schema checks.',

    },

    {

        'next_step': 'Monitor feature drift and prediction drift',

        'priority': 'High',

        'reason': 'Detect distribution shift early.',

        'owner_candidate': 'MLOps Engineer',

        'expected_output': 'Drift dashboards and alerting.',

    },

    {

        'next_step': 'Add scheduled retraining',

        'priority': 'Medium',

        'reason': 'Maintain performance over time.',

        'owner_candidate': 'ML Engineer',

        'expected_output': 'Retraining workflow with approval gates.',

    },

    {

        'next_step': 'Expose API only after validation',

        'priority': 'Medium',

        'reason': 'Prevent unvalidated deployment decisions.',

        'owner_candidate': 'Platform Team',

        'expected_output': 'Controlled model-serving endpoint.',

    },

    {

        'next_step': 'Validate what-if scenarios with metallurgists',

        'priority': 'High',

        'reason': 'Ensure domain-consistent interpretation.',

        'owner_candidate': 'Metallurgy Team',

        'expected_output': 'Scenario acceptance criteria and guardrails.',

    },

])

roadmap_path = REPORTS_METRICS_DIR / 'production_readiness_roadmap.csv'

roadmap_df.to_csv(roadmap_path, index=False)



level6_checklist_df = pd.DataFrame([

    {

        'Requirement': 'Track experiments, parameters and metrics',

        'Evidence in project': 'mlflow_runs_audit.csv, mlflow_leaderboard_audit.csv, mlruns/',

        'Status': '✅',

        'Notes': 'MLflow + local CSV fallback used.',

    },

    {

        'Requirement': 'Manage model versions',

        'Evidence in project': 'model_version_audit.csv, models/selected/*.pkl',

        'Status': '✅',

        'Notes': 'Selected/fallback/sensitivity/legacy statuses assigned.',

    },

    {

        'Requirement': 'Clearly identify selected model',

        'Evidence in project': 'selected_model_metadata.json summary table',

        'Status': '✅',

        'Notes': 'Primary model and fallback explicitly documented.',

    },

    {

        'Requirement': 'Ensure training reproducibility',

        'Evidence in project': 'config.yaml, metadata roles, feature lists',

        'Status': '✅',

        'Notes': 'Configuration and selected artifacts are traceable.',

    },

    {

        'Requirement': 'Ensure inference reproducibility',

        'Evidence in project': 'reproducibility_artifact_checklist.csv, feature_consistency_audit.csv',

        'Status': '✅',

        'Notes': 'Required artifacts and schema constraints audited.',

    },

    {

        'Requirement': 'Document assumptions',

        'Evidence in project': 'operational_assumptions_audit.csv',

        'Status': '✅',

        'Notes': 'Operational assumptions and mitigations formalized.',

    },

    {

        'Requirement': 'Document configurations',

        'Evidence in project': 'config.yaml and metadata references in audit',

        'Status': '✅',

        'Notes': 'Tracking URI, paths, and flags captured.',

    },

    {

        'Requirement': 'Document limitations',

        'Evidence in project': 'mlops_limitations_audit.csv, roadmap table',

        'Status': '✅',

        'Notes': 'Gaps and next actions clearly stated.',

    },

])

level6_checklist_path = REPORTS_METRICS_DIR / 'level6_checklist_audit.csv'

level6_checklist_df.to_csv(level6_checklist_path, index=False)



selected_row = metadata_summary_df.loc[

    metadata_summary_df['role'] == 'recommended_model_with_lagged_lab_assumption'

]

fallback_row = metadata_summary_df.loc[

    metadata_summary_df['role'] == 'strict_no_lab_input_fallback'

]



selected_model_name = selected_row['model_name'].iloc[0] if not selected_row.empty else 'N/A'

fallback_model_name = fallback_row['model_name'].iloc[0] if not fallback_row.empty else 'N/A'

best_val_rmse = selected_row['validation_RMSE'].iloc[0] if not selected_row.empty else 'N/A'

final_test_rmse = selected_row['test_RMSE'].iloc[0] if not selected_row.empty else 'N/A'

final_test_r2 = selected_row['test_R2'].iloc[0] if not selected_row.empty else 'N/A'



repro_status = 'PASS' if reproducibility_checklist_df['exists'].all() else 'PARTIAL'



summary_md = f'''# MLOps Audit Summary (Level 6)



## Selected model

- Selected model: {selected_model_name}

- Final role: recommended_model_with_lagged_lab_assumption



## Fallback model

- Fallback model: {fallback_model_name}

- Final role: strict_no_lab_input_fallback



## Metrics snapshot

- Best validation RMSE (selected model): {best_val_rmse}

- Final test RMSE (selected model): {final_test_rmse}

- Final test R2 (selected model): {final_test_r2}



## Core assumption

- The selected model assumes lagged laboratory quality is available at inference time.



## Artifacts generated by this audit

- mlflow_runs_audit.csv

- mlflow_leaderboard_audit.csv

- model_version_audit.csv

- reproducibility_artifact_checklist.csv

- selected_model_metadata_audit.csv

- feature_consistency_audit.csv

- operational_assumptions_audit.csv

- mlops_limitations_audit.csv

- production_readiness_roadmap.csv

- level6_checklist_audit.csv



## Reproducibility status

- Artifact checklist status: {repro_status}



## Limitations

- MLflow is local (not centralized governance).

- Model Registry stages/aliases are not implemented in this prototype.

- No automatic retraining or online drift monitoring yet.

- Scenario outputs remain predictive, not causal recommendations.



## Next steps

- Centralize MLflow backend.

- Implement formal Model Registry.

- Validate lab delay SLA with operations.

- Build monitored inference + retraining workflows.

'''



mlops_summary_path = REPORTS_METRICS_DIR / 'mlops_summary.md'

with open(mlops_summary_path, 'w', encoding='utf-8') as f:

    f.write(summary_md)



print(f'Assumptions table saved to: {assumptions_path}')

print(f'Limitations table saved to: {limitations_path}')

print(f'Roadmap table saved to: {roadmap_path}')

print(f'Level 6 checklist saved to: {level6_checklist_path}')

print(f'MLOps summary markdown saved to: {mlops_summary_path}')



display(assumptions_df)

display(limitations_df)

display(roadmap_df)

display(level6_checklist_df)
